In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv
/kaggle/input/datasets/mrinalpandey2/mcq-finall/adapter_model.safetensors
/kaggle/input/datasets/mrinalpandey2/mcq-finall/adapter_config.json


In [2]:
!pip install -Uq "trl[peft]" bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 38.7 MB/s eta 0:00:00


In [3]:
# ════════════════════════════════════════════════════════════════════════════
# ALL fast setup lives here: imports, paths, data, constants, prompt helpers.
# If the kernel restarts for any reason, just re-run THIS cell + cell 3.
# ════════════════════════════════════════════════════════════════════════════

import os, gc
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# ── paths ─────────────────────────────────────────────────────────────────────
BASE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
LORA_PATH     = "/kaggle/input/datasets/mrinalpandey2/mcq-finall"   # your uploaded dataset
COMP_PATH     = "/kaggle/input/competitions/smart-mcq-solver-challenge"

# ── data ──────────────────────────────────────────────────────────────────────
train = pd.read_csv(f"{COMP_PATH}/train.csv")
test  = pd.read_csv(f"{COMP_PATH}/test.csv")
print(f"train: {train.shape}  |  test: {test.shape}")
print("Answer distribution:\n", train['answer'].value_counts())

# ── constants ─────────────────────────────────────────────────────────────────
OPTION_LETTERS = ["A", "B", "C", "D", "E"]

PREFIXES = [
    "Pick the best possible answer:",
    "Determine the correct option:",
    "Select the most accurate option:",
    "Identify the correct statement:",
]

FIXED_PERMS = [
    ["A","B","C","D","E"],
    ["B","C","D","E","A"],
    ["C","D","E","A","B"],
    ["D","E","A","B","C"],
    ["E","A","B","C","D"],
]

# ── prompt helpers ────────────────────────────────────────────────────────────
def clean_prompt(prompt):
    prompt = prompt.strip()
    for p in PREFIXES:
        if prompt.lower().startswith(p.lower()):
            return prompt[len(p):].strip()
    return prompt

def build_prompt(row):
    q    = clean_prompt(row['prompt'])
    opts = '\n'.join(f"{l}. {row[l]}" for l in OPTION_LETTERS)
    return (
        "You are an expert at solving multiple-choice questions.\n\n"
        "Read the question carefully and choose the single best answer.\n\n"
        f"Question:\n{q}\n\n"
        f"Options:\n{opts}\n\n"
        "Respond with exactly one uppercase letter: A, B, C, D, or E.\n\n"
        "Answer:"
    )

def build_plm_prompt(row, perm):
    q    = clean_prompt(row['prompt'])
    opts = '\n'.join(
        f"{pos}. {row[orig]}"
        for pos, orig in zip(OPTION_LETTERS, perm)
    )
    return (
        "You are an expert at solving multiple-choice questions.\n\n"
        "Read the question carefully and choose the single best answer.\n\n"
        f"Question:\n{q}\n\n"
        f"Options:\n{opts}\n\n"
        "Respond with exactly one uppercase letter: A, B, C, D, or E.\n\n"
        "Answer:"
    )

# ── MAP@3 ─────────────────────────────────────────────────────────────────────
def map_at_3(preds, labels):
    scores = []
    for pred, label in zip(preds, labels):
        p = pred.strip().split()[:3]
        hit = [1.0 / (i + 1) for i, l in enumerate(p) if l == label]
        scores.append(hit[0] if hit else 0.0)
    return float(np.mean(scores))

# ── few-shot selection ────────────────────────────────────────────────────────
def row_max_opt_len(r):
    return max(len(str(r[l])) for l in OPTION_LETTERS)

short_train = train[train.apply(row_max_opt_len, axis=1) < 80].reset_index(drop=True)

def pick_fewshot(answer_letter, seed=0):
    pool = short_train[short_train['answer'] == answer_letter]
    if len(pool) == 0:
        pool = short_train
    return pool.sample(1, random_state=seed).iloc[0]

few_shot_rows = [
    pick_fewshot('A', seed=7),
    pick_fewshot('C', seed=3),
    pick_fewshot('E', seed=11),
]

FS_TURNS = []
for r in few_shot_rows:
    FS_TURNS.append({"role": "user",      "content": build_prompt(r)})
    FS_TURNS.append({"role": "assistant", "content": str(r['answer']).strip()})

print(f"Few-shot answers: {[r['answer'] for r in few_shot_rows]}")
print("Setup complete — ready to load model.")


train: (2000, 8)  |  test: (500, 7)
Answer distribution:
 answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Few-shot answers: ['A', 'C', 'E']
Setup complete — ready to load model.


In [4]:
# ── tokenizer from BASE MODEL ────────────────────────────────────────────────
# LoRA never changes the tokenizer vocabulary. Loading from LORA_PATH fails
# because save_pretrained() only writes config wrappers, not the tiktoken vocab.
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # left-pad for batched causal scoring
print("Tokenizer loaded")

# ── base model in 4-bit across both T4s ──────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print("Loading base model (this takes 2-3 min) …")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
    torch_dtype=torch.bfloat16,
)

# ── attach LoRA (only needs adapter_config.json + adapter_model.safetensors) ──
print("Attaching LoRA adapter …")
model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()
model.config.use_cache = True

for i in range(torch.cuda.device_count()):
    g = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {g.name}  {g.total_memory/1e9:.0f} GB  |  "
          f"reserved: {torch.cuda.memory_reserved(i)/1e9:.1f} GB")

# ── resolve single-token ids for A B C D E ───────────────────────────────────
def letter_token_id(letter):
    for candidate in (letter, " " + letter):
        ids = tokenizer.encode(candidate, add_special_tokens=False)
        if len(ids) == 1:
            return ids[0]
    return tokenizer.encode(letter, add_special_tokens=False)[-1]

option_token_ids = {l: letter_token_id(l) for l in OPTION_LETTERS}
print("Option token ids:", option_token_ids)


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Attaching LoRA adapter …
GPU 0: Tesla T4  16 GB  |  reserved: 2.0 GB
GPU 1: Tesla T4  16 GB  |  reserved: 6.8 GB
Option token ids: {'A': 32, 'B': 33, 'C': 34, 'D': 35, 'E': 36}


In [5]:
@torch.no_grad()
def score_plm_raw(rows, batch_size=8, perms=FIXED_PERMS, use_fewshot=True):
    """
    Accumulates RAW (pre-softmax) option logits across all permutations.
    Temperature is NOT applied here — call raw_to_preds(T=...) afterwards.
    This lets calibration run the model ONCE and sweep temperature in numpy.
    """
    all_raw = [{l: 0.0 for l in OPTION_LETTERS} for _ in rows]
    dev     = next(model.parameters()).device
    opt_ids = torch.tensor([option_token_ids[l] for l in OPTION_LETTERS], device=dev)

    for perm in tqdm(perms, desc="Permutations", leave=True):
        for start in tqdm(range(0, len(rows), batch_size), desc="Batches", leave=False):
            batch = rows[start:start + batch_size]
            idx   = list(range(start, min(start + batch_size, len(rows))))

            if use_fewshot:
                msgs_list = [
                    FS_TURNS + [{"role": "user", "content": build_plm_prompt(row, perm)}]
                    for row in batch
                ]
            else:
                msgs_list = [
                    [{"role": "user", "content": build_plm_prompt(row, perm)}]
                    for row in batch
                ]

            texts = [
                tokenizer.apply_chat_template(
                    msgs, tokenize=False, add_generation_prompt=True,
                )
                for msgs in msgs_list
            ]
            enc = tokenizer(
                texts, return_tensors="pt", padding=True,
                truncation=True, max_length=1024,
            ).to(dev)

            raw_logits = model(**enc).logits[:, -1, :]   # (B, vocab)
            opt_logits = raw_logits[:, opt_ids]           # (B, 5) — no softmax

            for bi, gi in enumerate(idx):
                for pi, pos_letter in enumerate(OPTION_LETTERS):
                    orig = perm[pi]
                    all_raw[gi][orig] += opt_logits[bi, pi].item()

    return all_raw


def raw_to_preds(raw_scores, temperature=1.0):
    """Pure numpy — converts accumulated raw logits to top-3 prediction strings."""
    preds = []
    for s in raw_scores:
        arr  = np.array([s[l] for l in OPTION_LETTERS], dtype=np.float64)
        arr  = arr / temperature
        arr -= arr.max()
        probs = np.exp(arr); probs /= probs.sum()
        ranked = [OPTION_LETTERS[i] for i in np.argsort(probs)[::-1]]
        preds.append(" ".join(ranked[:3]))
    return preds


In [6]:
# Model runs ONCE on 100 rows. Temperature sweep is instant numpy.
CALIB_N    = 100
CALIB_SEED = 99

fs_indices = {r.name for r in few_shot_rows}
calib_pool = train[~train.index.isin(fs_indices)]
calib_df   = calib_pool.sample(CALIB_N, random_state=CALIB_SEED).reset_index(drop=True)
calib_rows   = [row for _, row in calib_df.iterrows()]
calib_labels = calib_df['answer'].tolist()

print(f"Step 1/2 — model pass on {CALIB_N} calibration rows …")
calib_raw = score_plm_raw(calib_rows, batch_size=8)

print("Step 2/2 — temperature sweep (numpy only, no GPU) …")
TEMPERATURES = [0.3, 0.5, 0.7, 1.0, 1.3, 1.5]
best_temp, best_map = 1.0, -1.0

for T in TEMPERATURES:
    preds = raw_to_preds(calib_raw, temperature=T)
    m     = map_at_3(preds, calib_labels)
    print(f"  T={T:.1f}  MAP@3={m:.5f}")
    if m > best_map:
        best_map, best_temp = m, T

print(f"\nBest temperature : {best_temp}")
print(f"Calib MAP@3      : {best_map:.5f}")


Step 1/2 — model pass on 100 calibration rows …


Permutations:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Step 2/2 — temperature sweep (numpy only, no GPU) …
  T=0.3  MAP@3=0.97500
  T=0.5  MAP@3=0.97500
  T=0.7  MAP@3=0.97500
  T=1.0  MAP@3=0.97500
  T=1.3  MAP@3=0.97500
  T=1.5  MAP@3=0.97500

Best temperature : 0.3
Calib MAP@3      : 0.97500


In [7]:
print(f"Running PLM inference on {len(test)} test rows …")
print(f"  5 permutations × batch_size=8  |  temperature={best_temp}")

test_rows   = [row for _, row in test.iterrows()]
test_raw    = score_plm_raw(test_rows, batch_size=8)
predictions = raw_to_preds(test_raw, temperature=best_temp)

submission = pd.DataFrame({
    "ID"        : test["id"],
    "Prediction": predictions,
})

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
submission.head(10)


Running PLM inference on 500 test rows …
  5 permutations × batch_size=8  |  temperature=0.3


Permutations:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Saved submission.csv


,ID,Prediction
0,1,A D B
1,2,B D A
2,3,B E C
3,4,E A C
4,5,C D A
5,6,D A C
6,7,E A D
7,8,B E A
8,9,C D E
9,10,B C D


In [8]:
top1 = [p.split()[0] for p in predictions]
print("Top-1 distribution (should not be all one letter):")
print(dict(sorted(Counter(top1).items())))

assert len(submission) == len(test),                                "Row count mismatch"
assert submission['Prediction'].str.split().str.len().eq(3).all(), "Missing predictions"
print("All sanity checks passed.")


Top-1 distribution (should not be all one letter):
{'A': 87, 'B': 111, 'C': 116, 'D': 99, 'E': 87}
All sanity checks passed.
